In [ ]:
%pip install -U pageindex groq python-dotenv

In [31]:
import os, json, time
from dotenv import load_dotenv

load_dotenv()
PageIndex_API_KEY = os.getenv("PageIndex_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")


print("PageIndex key loaded:","✅" if PageIndex_API_KEY else "❌ Missing")
print("GROQ_API_KEY key loaded:","✅" if GROQ_API_KEY else "❌ Missing")

PageIndex key loaded: ✅
GROQ_API_KEY key loaded: ✅


In [7]:
from pageindex import PageIndexClient
from groq import Groq

pi_client = PageIndexClient(api_key=PageIndex_API_KEY)
groq_client = Groq(api_key=GROQ_API_KEY)

print("✅ PageIndex client is ready")
print("✅ GROQ client is ready")

✅ PageIndex client is ready
✅ GROQ client is ready


In [8]:
PDF_PATH = "./sample_document.pdf"

print(f"uploading ...  {PDF_PATH}")
result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]

print("Uploaded!")
print(f"Document ID : {doc_id}")
print("(save the ID, bcz it will be used all over the notebook)")

uploading ...  ./sample_document.pdf
Uploaded!
Document ID : pi-cmp4bw8mi00de01qwfdprmb41
(save the ID, bcz it will be used all over the notebook)


In [9]:
print("⏳ Building tree index...")
print("   (This runs once per document — the index is cached for reuse)")

while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f"   Status: {status}")
    
    if status == "completed":
        print("\n✅ Tree index ready!")
        break
    elif status == "failed":
        print("\n❌ Processing failed. Check your PDF format.")
        break
    
    time.sleep(5)

⏳ Building tree index...
   (This runs once per document — the index is cached for reuse)
   Status: processing
   Status: processing
   Status: processing
   Status: processing
   Status: processing
   Status: processing
   Status: processing
   Status: completed

✅ Tree index ready!


In [10]:
tree_result = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result",[])

print(f"📊 Top-level sections: {len(pageindex_tree)}")
print("\n🌲 Raw tree (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))


📊 Top-level sections: 25

🌲 Raw tree (first node):
{
  "title": "Preface",
  "node_id": "0000",
  "page_index": 1,
  "summary": "ZOOMNOTES FOR\nLINEAR ALGEBRA\n\nGILBERT STRANG\nMassachusetts Institute of Technology\n\nWELLESLEY - CAMBRIDGE PRESS\nBox 812060 Wellesley MA 02482\n",
  "text": "ZOOMNOTES FOR\nLINEAR ALGEBRA\n\nGILBERT STRANG\nMassachusetts Institute of Technology\n\nWELLESLEY - CAMBRIDGE PRESS\nBox 812060 Wellesley MA 02482\n"
}


In [11]:
def print_tree(nodes,indent=0):
    """Recursively print tree for the visual overview."""

    for node in nodes:
        prefix = " " * indent + ("└─" if indent > 0 else "")
        page = node.get("page_index","?")
        print(f"{prefix}[{node['node_id']}] {node['title']} (p.{page})")
        if node.get("nodes"):
            print_tree(node['nodes'], indent + 1)
    
print("📚 Full Document Structure:\n")
print_tree(pageindex_tree)


📚 Full Document Structure:

[0000] Preface (p.1)
[0001] Texts from Wellesley - Cambridge Press (p.2)
[0002] Content (p.3)
[0003] Preface (p.4)
[0004] Textbooks, ZoomNotes, and Video Lectures (p.5)
[0005] Three Great Factorizations: LU, QR, SVD (p.6)
[0006] Part 1 (p.7)
[0007] Basic Ideas of Linear Algebra (p.7)
 └─[0008] Part 1: Basic Ideas of Linear Algebra (p.8)
  └─[0009] 1.1 Linear Combinations of Vectors (p.8)
  └─[0010] 1.2 Dot Products  $\nu \cdot w$  and Lengths  $||\nu||$  and Angles  $\vartheta$ (p.9)
  └─[0011] 1.3 Matrices Multiplying Vectors (p.10)
  └─[0012] 1.4 Column Space and Row Space of $A$ (p.11)
  └─[0013] 1.5 Dependent and Independent Columns (p.12)
  └─[0014] 1.6 Matrix-Matrix Multiplication AB (p.13)
  └─[0015] 1.7 Factoring $A$ into $CR$: Column rank = $r$ = Row rank (p.14)
  └─[0016] 1.8 Rank one matrices $ A = (1 \text{ column}) $ times (1 row) (p.15)
[0017] Part 2 (p.16)
[0018] Solving Linear Equations (p.16)
 └─[0019] Part 2: Solving Linear Equations $A x =

In [12]:
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n['nodes'])

    return total


total = count_nodes(pageindex_tree)
print(f"🔢 Total nodes in tree: {total}")
print("   Each node = one retrievable section of the document")

🔢 Total nodes in tree: 42
   Each node = one retrievable section of the document


In [13]:
import json

def llm_tree_search(query: str, tree: list, model: str = "openai/gpt-oss-120b") -> dict:
    """
    Core PageIndex retrieval:
    Sends the query + document tree to the LLM.
    LLM responds with relevant node_ids.
    """

    # Compress tree to reduce tokens
    def compress(nodes):
        out = []

        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title": n["title"],
                "page": n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]
            }

            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])

            out.append(entry)

        return out

    compressed_tree = compress(tree)

    prompt = f"""
You are given a query and a document tree structure.

Your task:
1. Analyze the query carefully
2. Think step-by-step about which sections are relevant
3. Return the most relevant node IDs

Query:
{query}

Document Tree:
{json.dumps(compressed_tree, indent=2)}

Reply ONLY as valid JSON in this format:

{{
    "thinking": "your reasoning",
    "node_list": ["node_id1", "node_id2"]
}}
"""

    response = groq_client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],

        # Correct format for Groq
        response_format={"type": "json_object"},

        temperature=0
    )

    return json.loads(response.choices[0].message.content)

In [29]:
# ---- Test with a sample query------------------------------------
query = """what are four Fundamental Subspaces?"""
print(f"🔍 Query: {query}\n")
result = llm_tree_search(query,pageindex_tree)


print("🧠 LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("🎯 Selected Node IDs:", result.get("node_list", []))

🔍 Query: what are four Fundamental Subspaces?

🧠 LLM Reasoning:
The query asks for the four Fundamental Subspaces. The document nodes that explicitly mention 'Four Fundamental Subspaces' are node 0027 (title about Vector Spaces and Subspaces) and its child node 0028 which contains the detailed section. Additionally, node 0030 references the orthogonality of the four subspaces, which is also relevant. Therefore, the most relevant node IDs are 0027, 0028, and 0030.

🎯 Selected Node IDs: ['0027', '0028', '0030']


3 steps:

1. Tree Search → LLM picks relevant node_ids
2. Retrieve → Fetch the actual section content from those nodes
3. Generate → LLM writes a grounded answer with page citations

In [15]:
def find_nodes_by_ids(tree: list, target_ids: list):
    """Recursively walk the tree and collect nodes matching target_ids"""

    found = []
    for node in tree:
        if node['node_id']  in target_ids:
            found.append(node)
        if node.get("node"):
            found.extend(find_nodes_by_ids(node['nodes'],target_ids))
    return found

In [23]:
# ---- Generate answer from retrieved nodes ---------------------------

def generate_answer(query:str, nodes: list, model :str = "openai/gpt-oss-120b")-> str:
    """
    Takes retrieved nodes as context and generates a grounded answer
    Instructs the LLM to code the section titles and page numbers
    """
    if not nodes:
        return "⚠️ No relevant sections found in the document."
    
    # Build context string from retrieved nodes
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text','context not available')}"
        )
    context = "\n\n---\n\n".join(context_parts)
    prompt = f"""You are an expert document analyst.
Answer the question using ONLY the provided context.
For every claim you make, cite the section title and page number in parentheses.
Be concise and precise.
IMPORTANT FORMATTING RULES:
- Use proper mathematical notation
- Write equations in LaTeX
- Use Markdown math formatting:
    Inline math: $...$
    Block math: $$...$$
- Preserve formulas exactly
- Make answers visually clean and readable


Question: {query}

Context:
{context}

Answer:"""
    response  = response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user","content": prompt}]
    )

    return response.choices[0].message.content

In [24]:
# ---- The complete Vectorless RAG Function ------------------


def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG pipeline:
    
    Step 1: LLM Tree Search  → finds relevant node_ids
    Step 2: Node Retrieval   → fetches section content
    Step 3: Answer Generation → produces cited answer
    """
    if verbose:
        print(f"{'='*55}")
        print(f"🔍 Query: {query}")
        print(f"{'='*55}")
    
    # Step 1: Tree Search
    search_result  = llm_tree_search(query, tree)
    node_ids       = search_result.get("node_list", [])
    
    if verbose:
        print(f"\n🧠 Reasoning: {search_result.get('thinking', '')[:200]}...")
        print(f"🎯 Retrieved node IDs: {node_ids}")
    
    # Step 2: Retrieve nodes
    nodes = find_nodes_by_ids(tree, node_ids)
    
    if verbose:
        print(f"📄 Sections found: {[n['title'] for n in nodes]}")
    
    # Step 3: Generate answer
    answer = generate_answer(query, nodes)
    
    if verbose:
        print(f"\n📝 Answer:\n{answer}")
    
    return answer

In [25]:
# ── Run the full pipeline ────────────────────────────────────────────────────
answer = vectorless_rag(
    query="what are four Fundamental Subspaces?",
    tree=pageindex_tree
)

🔍 Query: what are four Fundamental Subspaces?

🧠 Reasoning: The query asks for the four Fundamental Subspaces. The document tree contains a section titled 'Vector Spaces and Four Fundamental Subspaces' under node 0027 and its child node 0028, which directly ad...
🎯 Retrieved node IDs: ['0027', '0028']
📄 Sections found: ['Vector Spaces and Subspaces Basis and Dimension']

📝 Answer:
The four fundamental subspaces associated with a matrix \(A\) are  

\[
\mathbf{C}(A),\;\mathbf{C}(A^{\mathrm{T}}),\;\mathbf{N}(A),\;\mathbf{N}(A^{\mathrm{T}}),
\]

i.e.:

* **Column space** \(\mathbf{C}(A)\) – the subspace spanned by the columns of \(A\).  
* **Row space** \(\mathbf{C}(A^{\mathrm{T}})\) – the column space of the transpose, equivalently the subspace spanned by the rows of \(A\).  
* **Nullspace** \(\mathbf{N}(A)\) – the set of solutions to \(A\mathbf{x}=0\).  
* **Left‑nullspace** \(\mathbf{N}(A^{\mathrm{T}})\) – the nullspace of the transpose, i.e., the set of vectors \(\mathbf{y}\) such tha

In [30]:
# ------- Test with Multiple queries --------------------
test_queries = [
    "how to find the cofactors?",
    "what is formula of  A^—1(A inverse)?",
    "give the examples of Eigenvalues and Eigenvectors?"
]
for q in test_queries:
    print()
    ans = vectorless_rag(q,pageindex_tree,verbose = False)
    print(f"Q: {q}")
    print(f"A: {ans[:300]}")
    print("-"*55)
    


Q: how to find the cofactors?
A: **Finding the cofactor \(C_{ij}\) of an entry \(a_{ij}\) in a square matrix \(A\)**  

1. **Delete the row \(i\) and column \(j\).**  
   The remaining \((n-1)\times (n-1)\) matrix is called the *minor* \(M_{ij}\).

2. **Take the determinant of the minor.**  
   \[
   \det(M_{ij})=\text{determinant 
-------------------------------------------------------

Q: what is formula of  A^—1(A inverse)?
A: The inverse of a square matrix \(A\) is obtained from its cofactor matrix \(C\) :

\[
\boxed{ \; A^{-1}= \frac{C^{\mathrm T}}{\det A}\; }
\]

where  

* \(C_{ij}=(-1)^{i+j}\det\bigl(M_{ij}\bigr)\) is the cofactor obtained by deleting row \(i\) and column \(j\) of \(A\), and  
* \(C^{\mathrm T}\) is 
-------------------------------------------------------

Q: give the examples of Eigenvalues and Eigenvectors?
A: **Example 1 – a 2 × 2 stochastic‑type matrix**

\[
A=\begin{pmatrix}0.8 & 0.3 \\[2pt] 0.2 & 0.7\end{pmatrix},
\qquad 
\det(A-\lambda I)=\lambda^{2}-1.